# 웨이퍼맵 결함 분류 — 2회차

**내 라벨로 학습시키기**

---

지난주에 여러분이 나눈 `my_labels.zip` 을 씁니다.

```
  ① 준비 (지난주와 같은 데이터를 다시 받습니다)
        ↓
  ② 내가 나눈 폴더를 올린다
        ↓
  ③ 세 가지로 학습한다   배포 200장 · 내 라벨 30장 · 둘을 합친 230장
        ↓
  ④ 셋을 똑같은 채점셋 50장으로 채점한다
        ↓
  ⑤ 애매한 이미지만 따로 모아 한 번 더 채점한다
```

> `my_labels.zip` 을 아직 못 만드셨다면 1회차 파일의 PART 6 을 먼저 하세요.

---
# PART 0 — 준비

**아래 셀을 위에서부터 순서대로 실행하세요.**
지난주와 같은 데이터와 같은 모델을 다시 준비하는 과정입니다. 읽지 않으셔도 됩니다.

In [ ]:
import os, glob, random, zipfile, shutil, re, hashlib
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

try:
    from google.colab import files
except Exception:
    files = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| 사용 장치:', device)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'gdown'],
               capture_output=True, text=True)
import gdown

ZIP_URL = 'https://drive.google.com/file/d/1NopA0gQpMMdWjsLxfjpwSjk-IYhoWQL9/view?usp=sharing'

DATA_DIR = '/content/wafer'
shutil.rmtree(DATA_DIR, ignore_errors=True)
os.makedirs(DATA_DIR, exist_ok=True)

JUNK = ('__MACOSX', '.DS_Store', 'Thumbs.db')

def clean_junk(base):
    """압축 프로그램이 끼워 넣는 파일을 지웁니다.
    맥에서 압축하면 ._사진.jpg 같은 가짜 이미지가 섞여 들어와 학습 중 에러가 납니다."""
    for p in sorted(glob.glob(os.path.join(base, '**', '*'), recursive=True), key=len, reverse=True):
        name = os.path.basename(p)
        if name in JUNK or name.startswith('._'):
            try:
                shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
            except OSError:
                pass

m = (re.search(r'/d/([A-Za-z0-9_-]{20,})', ZIP_URL)
     or re.search(r'[?&]id=([A-Za-z0-9_-]{20,})', ZIP_URL))
fid = m.group(1) if m else ZIP_URL.strip()
dst = '/content/wafer_set.zip'

ok = None
for attempt in (lambda: gdown.download(id=fid, output=dst, quiet=False),
                lambda: gdown.download(f'https://drive.google.com/uc?id={fid}', dst, quiet=False)):
    try:
        ok = attempt()
        if ok: break
    except TypeError:
        continue
    except Exception as e:
        print('  시도 실패:', type(e).__name__)

# 여기서 멈춰야 합니다. 그냥 넘어가면 한참 뒤에 엉뚱한 에러로 나타납니다.
if not (ok and zipfile.is_zipfile(dst)):
    raise RuntimeError('데이터를 받지 못했습니다.\n'
                       '  1) ZIP_URL 이 zip 파일 링크가 맞는지\n'
                       '  2) 공유 설정이 "링크가 있는 모든 사용자" 인지 확인하세요.')

with zipfile.ZipFile(dst) as z:
    z.extractall(DATA_DIR)
clean_junk(DATA_DIR)

print('\n준비 완료')
for d in sorted(glob.glob(os.path.join(DATA_DIR, '*'))):
    if os.path.isdir(d):
        n = len(glob.glob(os.path.join(d, '**', '*.*'), recursive=True))
        print(f'  {os.path.basename(d):16} {n}장')

In [ ]:
CLASS_NAMES = ['Donut', 'Scratch']

# Donut / Scratch 폴더가 어디에 있든 찾아냅니다.
def find_class_root(base):
    for h in sorted(glob.glob(os.path.join(base, '**', CLASS_NAMES[0]), recursive=True)):
        parts = h.replace(os.sep, '/').split('/')
        # testset 안에도 Donut / Scratch 가 있습니다. 그건 채점셋이므로 학습 폴더가 아닙니다.
        if not os.path.isdir(h) or '__MACOSX' in parts or 'testset' in parts:
            continue
        root = os.path.dirname(h)
        if all(os.path.isdir(os.path.join(root, c)) for c in CLASS_NAMES):
            return root
    return None

TRAIN_DIR = find_class_root(DATA_DIR)

if TRAIN_DIR is None:
    print('[확인 필요] Donut / Scratch 폴더를 찾지 못했습니다. 받은 내용:')
    for f in sorted(glob.glob(os.path.join(DATA_DIR, '**', '*'), recursive=True))[:20]:
        print('   ', os.path.relpath(f, DATA_DIR))
    raise FileNotFoundError('Donut / Scratch 폴더가 없습니다. 위 목록을 확인하세요.')

print('학습 폴더:', TRAIN_DIR)
for c in CLASS_NAMES:
    fs = glob.glob(os.path.join(TRAIN_DIR, c, '*.*'))
    print(f'  {c:10} {len(fs)}장')

# 한 장을 크게 봅니다 (원본 색 그대로)
sample = sorted(glob.glob(os.path.join(TRAIN_DIR, CLASS_NAMES[0], '*.*')))[0]
im = Image.open(sample).convert('RGB')
print(f'\n예시: {os.path.basename(sample)}  크기 {im.size}  (가로 x 세로 die 수)')
plt.figure(figsize=(4, 4))
plt.imshow(im, interpolation='nearest')
plt.axis('off'); plt.show()

In [ ]:
IMG_SIZE, BATCH_SIZE = 64, 16

# 웨이퍼맵은 배경/정상/불량 세 값만 있는 격자입니다.
# 크기를 바꿀 때 NEAREST를 써야 칸의 의미가 유지됩니다.
base_tf = [transforms.Grayscale(1),
           transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.NEAREST),
           transforms.ToTensor(),
           transforms.Normalize([0.5], [0.5])]
tf = transforms.Compose(base_tf)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, num_classes)
    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

print('준비 완료')

In [ ]:
class WaferFolder(datasets.ImageFolder):
    """Donut / Scratch 두 폴더만 클래스로 씁니다.
    (mixed 같은 폴더가 옆에 있어도 3번째 클래스로 끼어들지 않게 합니다.)"""
    def find_classes(self, directory):
        missing = [c for c in CLASS_NAMES if not os.path.isdir(os.path.join(directory, c))]
        if missing:
            raise FileNotFoundError(f'{directory} 안에 폴더가 없습니다: {missing}')
        return list(CLASS_NAMES), {c: i for i, c in enumerate(CLASS_NAMES)}


def run_cnn(folder, seed=42, epochs=15, quiet=False, eval_folder=None, exclude=()):
    """폴더를 읽어 CNN을 학습하고 결과를 돌려줍니다.

    eval_folder 를 주면 학습은 folder 로 하고 채점은 그 폴더로 합니다.
    exclude 는 채점에서 뺄 파일 이름 목록입니다.
    """
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)

    ds = WaferFolder(folder, transform=tf)
    ys = [y for _, y in ds.samples]
    counts = {c: ys.count(i) for i, c in enumerate(ds.classes)}
    if min(counts.values()) < 2:
        raise ValueError(f'각 폴더에 최소 2장은 있어야 합니다. 지금: {counts}')

    if eval_folder is None:
        tr, va = train_test_split(range(len(ds)), test_size=0.2,
                                  stratify=ys, random_state=seed)
        train_ds, val_ds = Subset(ds, tr), Subset(ds, va)
        val_src, val_idx = ds, va
    else:
        eds = WaferFolder(eval_folder, transform=tf)
        skip = set(exclude)
        keep = [i for i, (p, _) in enumerate(eds.samples) if os.path.basename(p) not in skip]
        train_ds, val_ds = ds, Subset(eds, keep)
        val_src, val_idx = eds, keep

    tl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    vl = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = SimpleCNN(len(ds.classes)).to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    for ep in range(1, epochs + 1):
        model.train()
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(x), y)
            loss.backward(); opt.step()
        if not quiet and ep % 5 == 0:
            print(f'  {ep:2d} epoch 진행')

    model.eval(); preds, tgts = [], []
    with torch.no_grad():
        for x, y in vl:
            preds += model(x.to(device)).argmax(1).cpu().tolist()
            tgts += y.tolist()

    return {
        'acc': accuracy_score(tgts, preds),
        'f1': f1_score(tgts, preds, average='macro'),
        'cm': confusion_matrix(tgts, preds, labels=list(range(len(ds.classes)))),
        'classes': ds.classes,
        'val_idx': val_idx, 'preds': preds, 'targets': tgts, 'ds': val_src,
        'n_train': len(train_ds), 'n_val': len(val_ds),
    }

def show_result(r, title=''):
    print(f"{title}  정확도 {r['acc']:.3f}   F1 {r['f1']:.3f}"
          f"   (학습 {r['n_train']}장 / 채점 {r['n_val']}장)")
    # 한쪽으로만 답한 경우를 알려줍니다
    if len(set(r['preds'])) == 1:
        only = r['classes'][r['preds'][0]]
        print(f"    [주의] 모델이 전부 '{only}' 라고만 답했습니다.")
        print("           정확도 숫자는 나왔지만 실제로는 아무것도 구분하지 못한 상태입니다.")
    cm, cls = r['cm'], r['classes']
    plt.figure(figsize=(3.6, 3.2))
    plt.imshow(cm, cmap='Blues')
    plt.xticks(range(len(cls)), cls); plt.yticks(range(len(cls)), cls)
    plt.xlabel('Predicted'); plt.ylabel('True (folder)')
    for i in range(len(cls)):
        for j in range(len(cls)):
            plt.text(j, i, cm[i, j], ha='center', va='center')
    plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()

print('준비 완료')

# ---- 2회차용 - 학습 폴더와 채점 폴더를 완전히 갈라서 돌립니다 ----
TARGET_STEPS = 300      # 세 모델의 '가중치를 고치는 횟수' 를 이 값으로 맞춥니다

def fit_predict(train_folder, test_folder, seed=42, steps=TARGET_STEPS):
    """train_folder 로 학습하고 test_folder 전부를 채점합니다.

    학습 장수가 다르면 epoch 수가 같아도 가중치를 고치는 횟수가 달라집니다.
    (200장이면 한 epoch 에 13번, 30장이면 2번) 그러면 라벨 때문에 갈린 것인지
    학습을 덜 해서 갈린 것인지 알 수 없습니다. 그래서 횟수를 먼저 맞추고
    epoch 수를 거기서 거꾸로 계산합니다."""
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)

    tr = WaferFolder(train_folder, transform=tf)
    te = WaferFolder(test_folder, transform=tf)
    ys = [y for _, y in tr.samples]
    counts = {c: ys.count(i) for i, c in enumerate(CLASS_NAMES)}
    if min(counts.values()) < 2:
        raise ValueError(f'각 폴더에 최소 2장은 있어야 합니다. 지금: {counts}')

    tl = DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True)
    epochs = max(1, -(-steps // len(tl)))

    model = SimpleCNN(len(CLASS_NAMES)).to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        model.train()
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); crit(model(x), y).backward(); opt.step()

    model.eval(); probs = []
    with torch.no_grad():
        for x, _ in DataLoader(te, batch_size=BATCH_SIZE):
            probs += torch.softmax(model(x.to(device)), 1)[:, 0].cpu().tolist()

    return {'files': [os.path.basename(p) for p, _ in te.samples],
            'y': [y for _, y in te.samples], 'p': probs,      # p = Donut 일 확률
            'n_train': len(tr), 'epochs': epochs, 'steps': epochs * len(tl)}


def score(res, keep=None):
    """채점 결과 중 일부만 골라 점수를 냅니다. keep 은 파일 이름 집합입니다."""
    idx = [i for i, f in enumerate(res['files']) if keep is None or f in keep]
    y = [res['y'][i] for i in idx]
    p = [0 if res['p'][i] >= 0.5 else 1 for i in idx]       # 0=Donut, 1=Scratch
    return {'acc': accuracy_score(y, p), 'f1': f1_score(y, p, average='macro'),
            'cm': confusion_matrix(y, p, labels=[0, 1]), 'n': len(idx),
            'files': [res['files'][i] for i in idx], 'y': y, 'pred': p,
            'one_sided': len(set(p)) == 1}

In [ ]:
# 2회차에서 쓸 값
N_MIXED = 30

# 지난주에 나눠 받은 30장이 배포본 어디에 있는지 찾아 둡니다.
# 이 30장은 배포된 Donut / Scratch 200장과 한 장도 겹치지 않습니다.
# (라벨이 붙어 있지 않은 웨이퍼맵에서 뽑은 것이라 정답이 없습니다)
MIXED_DIR = next((d for d in glob.glob(os.path.join(DATA_DIR, '**', 'mixed'), recursive=True)
                  if os.path.isdir(d) and glob.glob(os.path.join(d, '*.*'))), None)

# 여기서 멈춰야 합니다. 그냥 넘어가면 한참 뒤 PART 2 에서 엉뚱한 에러로 나타납니다.
if MIXED_DIR is None:
    print('[확인 필요] 배포본에서 mixed 폴더를 찾지 못했습니다. 받은 내용:')
    for f in sorted(glob.glob(os.path.join(DATA_DIR, '**', '*'), recursive=True))[:20]:
        print('   ', os.path.relpath(f, DATA_DIR))
    raise FileNotFoundError('mixed 폴더가 없습니다. PART 0 의 데이터 받기 셀부터 다시 실행하세요.')

MIXED_FILES = sorted(os.path.basename(p) for p in glob.glob(os.path.join(MIXED_DIR, '*.*')))

# 새 채점셋도 찾아 둡니다. 배포 200장과도, 위 30장과도 겹치지 않는 별개 웨이퍼입니다.
TESTSET_DIR = next((d for d in glob.glob(os.path.join(DATA_DIR, '**', 'testset'), recursive=True)
                    if os.path.isdir(d)
                    and all(os.path.isdir(os.path.join(d, c)) for c in CLASS_NAMES)), None)

if TESTSET_DIR is None:
    print('[확인 필요] 배포본에서 testset 폴더를 찾지 못했습니다. 받은 내용:')
    for f in sorted(glob.glob(os.path.join(DATA_DIR, '*')))[:20]:
        print('   ', os.path.basename(f))
    raise FileNotFoundError('testset 폴더가 없습니다. 배포본이 예전 판입니다. 강사에게 알려주세요.')

# 오늘 쓰는 폴더는 둘입니다. 서로 다른 웨이퍼이고, 한 장도 겹치지 않습니다.
_n_test = {c: len(glob.glob(os.path.join(TESTSET_DIR, c, '*.*'))) for c in CLASS_NAMES}

print('오늘 쓰는 폴더는 둘입니다')
print()
print(f'  [1] 지난주에 나눠 받은 {len(MIXED_FILES)}장   <- 학습에 씁니다 (내 라벨 모델)')
print(f'      {MIXED_DIR}')
print(f'      {", ".join(MIXED_FILES[:5])} ...')
print()
print(f'  [2] 채점셋 {sum(_n_test.values())}장   <- 오늘 점수를 매길 때만 씁니다')
print(f'      {TESTSET_DIR}')
print('      ' + '   '.join(f'{c} {n}장' for c, n in _n_test.items()))
print()
print('  둘은 다른 웨이퍼입니다. [2] 로 학습하지 않고, [1] 로 채점하지 않습니다.')

# 1회차에는 30장을 만드는 경로가 둘 있었습니다. 아래는 그중 '대체 경로' 를 되살리는 것입니다.
# (배포 zip 에 mixed 가 없던 시기에 1회차를 돌린 분은 이쪽 30장을 받으셨습니다.)
MIXED_DIR_DIST = MIXED_DIR          # (가) 배포된 mixed 30장

def build_fallback_30(out='/content/mixed_fallback'):
    """(나) 학습셋 200장에서 30장을 복사하던 경로를 그대로 되살립니다.

    1회차와 같은 난수 씨앗(0)을 쓰므로 어떤 파일이 뽑혔는지 정확히 재현됩니다."""
    rng = random.Random(0)
    picked = []
    for c in CLASS_NAMES:
        fs = sorted(glob.glob(os.path.join(TRAIN_DIR, c, '*.*')))
        picked += [(f, c) for f in rng.sample(fs, N_MIXED // len(CLASS_NAMES))]
    rng.shuffle(picked)

    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(out, exist_ok=True)
    m = {}
    for i, (src, c) in enumerate(picked, 1):
        name = f'wafer_{i:03d}{os.path.splitext(src)[1]}'
        shutil.copy(src, os.path.join(out, name))
        m[name] = (os.path.basename(src), c)      # 원본 파일과 원본 라벨
    return out, m


---
# PART 1 — 내가 나눈 폴더 올리기

아래 셀을 실행하면 파일 선택 창이 뜹니다. **`my_labels.zip`** 을 고르세요.

> 창이 안 뜨거나 멈추면, 왼쪽 폴더 아이콘에 zip 을 끌어다 놓고
> 셀 맨 위 `MY_ZIP` 에 경로를 적은 뒤 다시 실행하세요.

In [ ]:
# 나눈 결과(my_labels.zip)를 올립니다
#
# 파일 선택 창이 안 뜨거나 멈추는 브라우저가 있습니다. 그럴 때만 아래 한 줄을 쓰세요.
#   1) 왼쪽 폴더 아이콘에 my_labels.zip 을 끌어다 놓고
#   2) MY_ZIP 에 그 파일 경로를 적은 뒤 이 셀을 다시 실행
MY_ZIP = ''          # 예: MY_ZIP = '/content/my_labels.zip'

MY_DIR = '/content/my_labels'
MY_TRAIN = None
shutil.rmtree(MY_DIR, ignore_errors=True)
os.makedirs(MY_DIR, exist_ok=True)

if MY_ZIP:
    if not os.path.isfile(MY_ZIP):
        print('[확인 필요] 그 경로에 파일이 없습니다:', MY_ZIP)
        print('  /content 에 있는 zip 파일들:')
        for _p in sorted(glob.glob('/content/*.zip')):
            print('   ', _p)
        raise FileNotFoundError(MY_ZIP)
    src_zip = MY_ZIP
    print('직접 지정한 파일을 씁니다:', src_zip)

elif files is not None:
    print('파일 선택 창이 뜹니다. my_labels.zip 을 고르세요.')
    up = files.upload()
    zips = [n for n in up if n.lower().endswith('.zip')]
    if not zips:
        raise RuntimeError('zip 파일이 아닙니다. my_labels 폴더째 압축해서 올리세요.')
    src_zip = zips[0]
    print('올린 파일:', src_zip)

else:
    raise RuntimeError('왼쪽 폴더 아이콘으로 my_labels.zip 을 올리고, '
                       '이 셀 맨 위 MY_ZIP 에 경로를 적은 뒤 다시 실행하세요.')

with zipfile.ZipFile(src_zip) as z:
    z.extractall(MY_DIR)

clean_junk(MY_DIR)   # 맥 압축 찌꺼기(._*.jpg) 제거 - 남으면 학습 중 에러가 납니다

# 폴더 구조 정리 - 압축 방식에 따라 한 겹 더 들어가 있을 수 있습니다
found = {}
for c in CLASS_NAMES:
    hits = [h for h in sorted(glob.glob(os.path.join(MY_DIR, '**', c), recursive=True))
            if os.path.isdir(h)]
    if hits:
        found[c] = hits[0]

same_root = (len(found) == len(CLASS_NAMES)
             and len({os.path.dirname(v) for v in found.values()}) == 1)

if same_root:
    MY_TRAIN = os.path.dirname(found[CLASS_NAMES[0]])
    for c in CLASS_NAMES:
        print(f'{c:10} {len(glob.glob(os.path.join(found[c], "*.*")))}장')
    print('\n사용할 경로:', MY_TRAIN)
else:
    print('[확인 필요] Donut / Scratch 폴더를 같은 위치에서 찾지 못했습니다.')
    print('  my_labels 폴더 안에 Donut, Scratch 두 폴더가 나란히 있어야 합니다.')
    for f in sorted(glob.glob(os.path.join(MY_DIR, '**', '*'), recursive=True))[:15]:
        print('  ', os.path.relpath(f, MY_DIR))

In [ ]:
# 올린 폴더를 확인합니다. 배포된 30장이 그대로 들어 있어야 합니다.
if not MY_TRAIN:
    raise RuntimeError('위 셀에서 my_labels.zip 업로드가 정상으로 끝나야 합니다.')

MY_LABEL = {}          # {'wafer_001.jpg': '내가 넣은 폴더 이름'}
for c in CLASS_NAMES:
    for p in glob.glob(os.path.join(MY_TRAIN, c, '*.*')):
        MY_LABEL[os.path.basename(p)] = c

def _pix(path):
    """이미지 자체의 지문. 파일 이름이 같아도 그림이 다르면 값이 달라집니다."""
    return hashlib.md5(np.array(Image.open(path).convert('RGB')).tobytes()).hexdigest()

def _match_count(folder):
    ref = {os.path.basename(p): _pix(p) for p in glob.glob(os.path.join(folder, '*.*'))}
    return sum(1 for c in CLASS_NAMES
               for p in glob.glob(os.path.join(MY_TRAIN, c, '*.*'))
               if os.path.basename(p) in ref and _pix(p) == ref[os.path.basename(p)])

MIXED_DIR = MIXED_DIR_DIST          # 배포된 mixed 30장 - 이것이 기본입니다
MIX_MAP = None
MIXED_FILES = sorted(os.path.basename(p) for p in glob.glob(os.path.join(MIXED_DIR, '*.*')))
_same = _match_count(MIXED_DIR)

_cnt = {c: sum(1 for v in MY_LABEL.values() if v == c) for c in CLASS_NAMES}
print(f'올린 장수 {len(MY_LABEL)}장 / 배포된 {len(MIXED_FILES)}장')
print('  ' + '    '.join(f'{c} {n}장' for c, n in _cnt.items()))

# ---- 여기서부터는 뭔가 어긋났을 때만 나옵니다 ----
if _same < len(MIXED_FILES):
    # 1회차에 배포 zip 에 mixed 가 없어 '대체 경로' 로 받으신 경우가 있습니다.
    # 그때 30장은 학습셋에서 복사된 다른 웨이퍼인데 파일 이름이 똑같습니다.
    _fb_dir, _fb_map = build_fallback_30()
    _fb_same = _match_count(_fb_dir)
    if _fb_same > _same:
        MIXED_DIR, MIX_MAP, _same = _fb_dir, _fb_map, _fb_same
        MIXED_FILES = sorted(os.path.basename(p) for p in glob.glob(os.path.join(MIXED_DIR, '*.*')))
        print('\n  [안내] 1회차 대체 경로로 받은 30장을 나누셨습니다.')
        print('         학습셋에서 복사된 것이라 원래 라벨이 있습니다.')
        _agree = sum(1 for n, c in MY_LABEL.items() if n in MIX_MAP and MIX_MAP[n][1] == c)
        print(f'         내 라벨과 원래 라벨이 같은 것: {_agree}/{len(MY_LABEL)}장')

if _same == 0:
    raise RuntimeError(
        '올린 이미지가 배포본과 맞지 않습니다.\n'
        '  1회차 노트북 PART 6 을 다시 실행해 mixed.zip 을 새로 받아 나눠 주세요.\n'
        '  (파일 이름을 바꾸셨거나, 다른 30장을 나누신 경우입니다)')

_miss = sorted(set(MIXED_FILES) - set(MY_LABEL))
_extra = sorted(set(MY_LABEL) - set(MIXED_FILES))

if _same < len(MY_LABEL):
    print(f'\n  [확인 필요] {len(MY_LABEL) - _same}장은 이름은 같은데 그림이 다릅니다.')
    print('    여러 벌이 섞였을 수 있습니다. 받은 30장 한 벌로만 다시 나눠 주세요.')
if _miss:
    print(f'\n  [확인 필요] 빠진 파일 {len(_miss)}장:',
          ', '.join(_miss[:8]) + (' ...' if len(_miss) > 8 else ''))
    print('    30장을 다 나눠야 합니다.')
if _extra:
    print(f'\n  [확인 필요] 모르는 파일 {len(_extra)}장:', ', '.join(_extra[:8]))
    print('    받은 파일 이름을 바꾸면 안 됩니다.')
if min(_cnt.values()) < 2:
    raise RuntimeError(f'한 폴더에 최소 2장은 있어야 학습이 됩니다. 지금: {_cnt}')

print(f'\n확인 완료 - 나눠 받은 30장이 그대로 들어 있습니다 ({_same}/{len(MIXED_FILES)}장)')

---
## 내가 나눈 결과를 한 화면에서 보기

**숫자를 보기 전에 이것부터 봅니다.**

한 장씩 볼 때는 몰랐던 것이, 같은 폴더에 넣은 것끼리 나란히 놓으면 보입니다.
내가 Donut 이라고 넣은 것들이 서로 닮았습니까?

In [ ]:
N = 6
fig, axes = plt.subplots(2, N, figsize=(2.2 * N, 5.0))
for r, c in enumerate(CLASS_NAMES):
    fs = sorted(glob.glob(os.path.join(MY_TRAIN, c, '*.*')))[:N]
    print(f'{c:10} {len(glob.glob(os.path.join(MY_TRAIN, c, "*.*")))}장 중 앞 {len(fs)}장')
    for k in range(N):
        ax = axes[r, k]; ax.axis('off')
        if k < len(fs):
            ax.imshow(Image.open(fs[k]).convert('RGB'), interpolation='nearest')
            ax.set_title(c + '  ' + os.path.basename(fs[k]), fontsize=8)
plt.tight_layout(); plt.show()

---
## 내 라벨과 강사 라벨을 맞춰봅니다

이 30장에는 **정답이 없습니다.** 아무도 라벨을 붙이지 않은 이미지였습니다.

그래서 아래에서 맞춰보는 것은 정답이 아니라 **강사가 같은 30장을 나눈 결과**입니다.
같은 이미지를 놓고 두 사람이 **어디서 갈렸는지**를 보는 것이지,
누가 맞고 누가 틀렸는지를 보는 것이 아닙니다.

> 갈린 장의 파일 이름을 적어 두세요. 수업에서 그 이미지를 띄웁니다.

In [ ]:
# 강사가 같은 30장을 나눈 결과와 맞춰봅니다.
# 이것은 정답표가 아닙니다. '또 한 사람의 판단' 입니다.
#
# (강사용) 아래 두 줄에 번호만 적어서 배포하십시오. 비워 두면 이 셀은 그냥 넘어갑니다.
REF_DONUT   = [2,4,5,8,9,13,15,16,17,19,22,25,26,27,28]
REF_SCRATCH = [1,3,6,7,10,11,12,14,18,20,21,23,24,29,30]

REF_LABEL = {f'wafer_{n:03d}.jpg': 'Donut' for n in REF_DONUT}
REF_LABEL.update({f'wafer_{n:03d}.jpg': 'Scratch' for n in REF_SCRATCH})

if not REF_LABEL:
    print('강사 라벨이 아직 들어 있지 않습니다. 이 셀은 건너뜁니다.')
    print('(강사님: 이 셀 위쪽 REF_DONUT / REF_SCRATCH 에 번호를 채워 배포하세요)')
else:
    _pair = [(n, REF_LABEL[n], MY_LABEL[n]) for n in sorted(REF_LABEL) if n in MY_LABEL]
    _same = [t for t in _pair if t[1] == t[2]]
    _diff = [t for t in _pair if t[1] != t[2]]

    print(f'맞춰본 장수 {len(_pair)}장')
    print(f'  같게 봤음    {len(_same):2d}장  ({len(_same) / max(1, len(_pair)):.0%})')
    print(f'  다르게 봤음  {len(_diff):2d}장')

    if _diff:
        _n = min(len(_diff), 6)
        print(f'\n갈린 것 {len(_diff)}장 중 앞 {_n}장 - 파일 이름을 적어 두세요:')
        print('  ', ', '.join(t[0] for t in _diff))
        fig, axes = plt.subplots(1, _n, figsize=(2.4 * _n, 3.2))
        if _n == 1:
            axes = [axes]
        for ax, (name, ref, mine) in zip(axes, _diff[:_n]):
            ax.imshow(Image.open(os.path.join(MIXED_DIR, name)).convert('RGB'),
                      interpolation='nearest')
            # ref = 강사가 넣은 폴더, mine = 내가 넣은 폴더
            ax.set_title(f'{name}\nref {ref}   mine {mine}', fontsize=8)
            ax.axis('off')
        plt.tight_layout(); plt.show()
        print('다르게 본 것이 몇 장이든 그것은 오답이 아닙니다.')
        print('정답이 없는 이미지이므로, 갈렸다는 사실 자체가 오늘의 재료입니다.')
    else:
        print('\n30장 모두 같게 봤습니다. 그렇다고 둘 다 맞는다는 뜻은 아닙니다.')

---
# PART 2 — 세 가지로 학습해서 나란히 놓기

```
  A   배포 200장            1회차와 같은 방식. 남이 붙인 라벨
  B   내 라벨 30장          내가 나눈 그대로
  C   A + B = 230장         합치면 나아지는가
                 ↓
      똑같은 채점셋 50장으로 세 번 채점
```

**세 모델의 차이는 딱 하나입니다 — 어떤 폴더를 학습에 넣었는가.**
모델 구조, 학습 횟수, 채점셋, 채점 방식은 전부 같습니다.

### 채점셋이 학습셋과 다른 데이터입니다

이번 채점셋 50장은 배포된 200장에서 떼어낸 것이 아니라, **원본 데이터에서 따로 뽑아온
다른 웨이퍼**입니다. 배포 200장과도, 여러분이 나눈 30장과도 한 장도 겹치지 않습니다.

그래서 여기 나오는 숫자는 **"배운 것과 다른 데이터에서도 되는가"** 를 잰 값입니다.

지난주와 다른 숫자가 나올 겁니다. 높을 수도 있고 낮을 수도 있습니다.
**채점셋이 바뀌었으니 당연한 일입니다.** 그래서 "이 모델의 정확도는 0.9x" 라고 말할 수 없고,
**"이 채점셋에서 0.9x"** 라고만 말할 수 있습니다. 지난주 교훈이 여기서 한 번 더 나옵니다.

> 여러분이 나눈 30장에는 **정답이 없습니다.** 라벨이 안 붙어 있는 웨이퍼맵에서
> 모델이 가장 헷갈린 것만 골라낸 것이기 때문입니다.
> 그래서 "여러분의 라벨이 몇 점인지" 는 채점할 수 없습니다.
> 채점할 수 있는 것은 **그 라벨로 배운 모델이 다른 이미지를 얼마나 맞히는지** 입니다.

In [ ]:
# 학습 폴더와 채점 폴더가 애초에 갈라져 있습니다. 나눌 것이 없습니다.
#
#   학습   배포 200장        Wafermap/Donut,  Wafermap/Scratch     <- A 가 씁니다
#   채점   채점셋 50장       testset/Donut,   testset/Scratch      <- 세 모델 모두 이걸로
#
# 채점셋은 배포 200장과도, 여러분이 나눈 30장과도 한 장도 겹치지 않습니다.
# 아래에서 파일 이름이 아니라 '이미지 자체' 로 확인합니다. 이름은 얼마든지 같을 수 있습니다.

POOL_TRAIN = TRAIN_DIR            # A - 배포 200장 그대로
POOL_TEST = TESTSET_DIR           # 채점셋 50장

MY30 = '/content/my30'            # B - 내 라벨 30장
shutil.rmtree(MY30, ignore_errors=True)
for c in CLASS_NAMES:
    os.makedirs(os.path.join(MY30, c), exist_ok=True)
for _name, _c in MY_LABEL.items():
    if _name in MIXED_FILES:
        shutil.copy(os.path.join(MIXED_DIR, _name), os.path.join(MY30, _c, _name))

COMBINED = '/content/combined'    # C - 배포 200장 + 내 30장
shutil.rmtree(COMBINED, ignore_errors=True)
for c in CLASS_NAMES:
    os.makedirs(os.path.join(COMBINED, c), exist_ok=True)
    for p in glob.glob(os.path.join(TRAIN_DIR, c, '*.*')):
        shutil.copy(p, os.path.join(COMBINED, c, os.path.basename(p)))
    for p in glob.glob(os.path.join(MY30, c, '*.*')):
        shutil.copy(p, os.path.join(COMBINED, c, os.path.basename(p)))

def test_path(fname):
    """채점셋 안에서 파일을 찾습니다 (그림 그릴 때 씁니다)."""
    for c in CLASS_NAMES:
        p = os.path.join(POOL_TEST, c, fname)
        if os.path.isfile(p):
            return p

# 겹침 확인 - 이미지 자체로 봅니다
_s_train = {_pix(p) for c in CLASS_NAMES for p in glob.glob(os.path.join(TRAIN_DIR, c, '*.*'))}
_s_test = {_pix(p) for c in CLASS_NAMES for p in glob.glob(os.path.join(POOL_TEST, c, '*.*'))}
_s_mine = {_pix(p) for p in glob.glob(os.path.join(MIXED_DIR, '*.*'))}

print(f'학습 {len(_s_train)}장 / 채점 {len(_s_test)}장 / 내가 나눈 것 {len(_s_mine)}장')
print('  채점셋 ∩ 학습셋  :', len(_s_test & _s_train), '장   (0 이어야 합니다)')
print('  채점셋 ∩ 내 30장 :', len(_s_test & _s_mine), '장   (0 이어야 합니다)')
for _k, _v in [('A 배포', POOL_TRAIN), ('B 내 라벨', MY30), ('C 합침', COMBINED)]:
    print(f'  {_k:10}', {c: len(glob.glob(os.path.join(_v, c, '*.*'))) for c in CLASS_NAMES})

### 채점셋 50장을 두 덩어리로 가릅니다

50장을 한꺼번에 채점하면 **쉬운 것에 묻혀** 모델 차이가 안 보입니다.
그래서 애매한 것만 따로 모아 한 번 더 채점합니다.

```
  T-EASY   명확한 것    세 모델이 다 비슷하게 맞힙니다
  T-HARD   애매한 것    여기서만 차이가 벌어집니다
```

> **기준을 밝혀 둡니다.** "배포 200장으로 학습한 모델 5개의 평균 확률이 0.5 에 가까운 것"
> 을 애매하다고 봤습니다. 이것은 **모델이 헷갈린 것**이라는 뜻이고,
> 사람이 헷갈린 것과 같지 않습니다. 그리고 이 기준은 A·C 에게 불리합니다
> (A 같은 모델이 어려워한 것을 일부러 모았으니까요).
> 그래도 세 모델 모두 **같은 50장**으로 채점하므로 모델끼리 비교는 그대로 성립합니다.

In [ ]:
# 채점셋을 애매한 것 / 명확한 것으로 가릅니다 (모델 5개의 평균 확률로)
SEL_SEEDS = [0, 1, 2, 3, 4]

print('애매한 것을 고르는 중', end='')
_ens = []
for s in SEL_SEEDS:
    _ens.append(fit_predict(POOL_TRAIN, POOL_TEST, seed=s))
    print('.', end='')
print()

TEST_FILES = _ens[0]['files']
TEST_Y = _ens[0]['y']
_pbar = np.mean([e['p'] for e in _ens], axis=0)     # Donut 일 확률의 평균
_unc = np.abs(_pbar - 0.5)                          # 0 에 가까울수록 애매

T_HARD, T_EASY = set(), set()
for _ci in range(len(CLASS_NAMES)):
    _rows = sorted([i for i, y in enumerate(TEST_Y) if y == _ci], key=lambda i: _unc[i])
    _half = len(_rows) // 2                      # 클래스마다 절반씩 (채점셋 크기가 바뀌어도 됩니다)
    T_HARD |= {TEST_FILES[i] for i in _rows[:_half]}
    T_EASY |= {TEST_FILES[i] for i in _rows[_half:]}

print(f'T-HARD(애매) {len(T_HARD)}장   T-EASY(명확) {len(T_EASY)}장   겹침 {len(T_HARD & T_EASY)}장')

# 그래프 안에는 한글을 쓰지 않습니다 (Colab 에 한글 폰트가 없어 네모로 깨집니다)
_order = np.argsort(_unc)
for _title, _idxs in [('T-HARD : most ambiguous 6', _order[:6]),
                      ('T-EASY : most clear 6', _order[::-1][:6])]:
    fig, axes = plt.subplots(1, 6, figsize=(13.5, 2.9))
    for ax, i in zip(axes, _idxs):
        ax.imshow(Image.open(test_path(TEST_FILES[i])).convert('RGB'), interpolation='nearest')
        ax.set_title(f'{CLASS_NAMES[TEST_Y[i]]}\np(Donut)={_pbar[i]:.2f}', fontsize=8)
        ax.axis('off')
    fig.suptitle(_title, fontsize=11)
    plt.tight_layout(); plt.show()

print('위 두 줄을 비교해 보세요. 아래쪽(명확)은 눈으로도 갈립니까.')

### 세 모델을 돌립니다

**학습 횟수를 맞춥니다.** 200장과 30장을 같은 epoch 수로 돌리면
가중치를 고치는 횟수가 6배 넘게 차이 납니다. 그러면 라벨 때문에 갈린 것인지
학습을 덜 해서 갈린 것인지 알 수 없습니다. 그래서 세 모델 모두
**같은 횟수(300번)** 로 맞추고 epoch 수를 거기서 거꾸로 계산했습니다.

그리고 각 모델을 **seed 4개**로 돌립니다. 지난주에 본 것 그대로 — 한 번 돌린 숫자는
"이번에 이렇게 나왔다"는 뜻일 뿐이라서, 네 번 돌려 **평균과 흔들림 폭**을 함께 봅니다.

> 1~3분 걸립니다.

In [ ]:
# 세 모델 x seed 4개 = 12번 학습합니다 (1~3분)
MODELS = [('A', '배포 200장',   POOL_TRAIN),
          ('B', '내 라벨 30장',  MY30),
          ('C', '합침 230장',   COMBINED)]
SEEDS = [42, 7, 123, 2024]

RES = {}
for _k, _label, _folder in MODELS:
    print(f'{_k}  {_label:14}', end=' ')
    for s in SEEDS:
        RES[(_k, s)] = fit_predict(_folder, POOL_TEST, seed=s)
        print('.', end='')
    _r = RES[(_k, SEEDS[0])]
    print(f'  학습 {_r["n_train"]:3d}장 / {_r["epochs"]:3d} epoch / 가중치 갱신 {_r["steps"]}회')
print('\n끝났습니다')

In [ ]:
# 표 - 세 모델을 같은 채점셋으로 나란히
import unicodedata

def _wide(s):
    """한글은 화면에서 두 칸을 차지합니다. 그 폭을 셉니다."""
    return sum(2 if unicodedata.east_asian_width(ch) in 'WF' else 1 for ch in s)

def _rpad(s, w):
    return s + ' ' * max(0, w - _wide(s))

def _lpad(s, w):
    return ' ' * max(0, w - _wide(s)) + s

N_ALL = f'전체 {len(TEST_FILES)}장'
SUBSETS = [(N_ALL, None), ('애매 T-HARD', T_HARD), ('명확 T-EASY', T_EASY)]

_head = _rpad('', 20) + ''.join(_lpad(n, 22) for n, _ in SUBSETS)
print(_head)
print('-' * _wide(_head))

SUMMARY = {}
for _k, _label, _ in MODELS:
    _row = _rpad(f'{_k}  {_label}', 20)
    for _name, _keep in SUBSETS:
        _a = [score(RES[(_k, s)], _keep)['acc'] for s in SEEDS]
        SUMMARY[(_k, _name)] = (float(np.mean(_a)), min(_a), max(_a))
        _row += _lpad(f'{np.mean(_a):.3f} ({min(_a):.2f}~{max(_a):.2f})', 22)
    _flag = sum(score(RES[(_k, s)])['one_sided'] for s in SEEDS)
    print(_row + (f'   <- 한쪽으로만 답한 seed {_flag}개' if _flag else ''))

print('\n괄호 안은 seed 4개의 최저~최고입니다.')

# 막대그래프 (그래프 안에는 한글을 쓰지 않습니다)
_labels = {'A': 'A dist200', 'B': 'B mine30', 'C': 'C both230'}
_legend = ['ALL %d' % len(TEST_FILES), 'HARD %d' % len(T_HARD), 'EASY %d' % len(T_EASY)]
fig, ax = plt.subplots(figsize=(8.5, 3.8))
_w, _x = 0.26, np.arange(len(MODELS))
for _j, (_name, _) in enumerate(SUBSETS):
    _m = [SUMMARY[(k, _name)][0] for k, _, _ in MODELS]
    _lo = [_m[i] - SUMMARY[(k, _name)][1] for i, (k, _, _) in enumerate(MODELS)]
    _hi = [SUMMARY[(k, _name)][2] - _m[i] for i, (k, _, _) in enumerate(MODELS)]
    ax.bar(_x + (_j - 1) * _w, _m, _w, yerr=[_lo, _hi], capsize=3, label=_legend[_j])
ax.axhline(0.5, ls='--', lw=1, color='gray')
ax.text(len(MODELS) - 0.55, 0.51, 'coin flip', fontsize=8, color='gray')
ax.set_xticks(_x); ax.set_xticklabels([_labels[k] for k, _, _ in MODELS])
ax.set_ylim(0.3, 1.0); ax.set_ylabel('accuracy'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

_a = SUMMARY[('A', '애매 T-HARD')][0]
_b = SUMMARY[('B', '애매 T-HARD')][0]
_c = SUMMARY[('C', '애매 T-HARD')][0]
print(f'애매한 {len(T_HARD)}장에서')
print(_rpad('  A 배포 200장', 20) + f'{_a:.3f}')
print(_rpad('  B 내 라벨 30장', 20) + f'{_b:.3f}   (A 와 차이 {_b - _a:+.3f})')
print(_rpad('  C 합침 230장', 20) + f'{_c:.3f}   (A 와 차이 {_c - _a:+.3f})')

# 숫자를 잘못 읽기 쉬운 두 경우를 미리 짚어 둡니다
_b_all = SUMMARY[('B', N_ALL)][0]
_b_easy = SUMMARY[('B', '명확 T-EASY')][0]

if _b_all < 0.45 or _b_easy < 0.45:
    print()
    print('[B 가 0.5 아래로 내려갔습니다]')
    print('  실력이 없다는 뜻이 아닙니다. 예측이 뒤집혔다는 뜻입니다.')
    print('  내가 Donut 이라고 부른 모양을 채점셋은 Scratch 라고 부르고 있다는 신호입니다.')
    print('  두 이름을 서로 바꾸면 점수가 올라갑니다. 즉 모양은 구분했고, 이름이 반대입니다.')
    print('  아래 혼동행렬에서 대각선이 아니라 반대 칸에 숫자가 몰려 있는지 확인하세요.')
elif _b_all < 0.65:
    print()
    print('[B 가 동전 던지기(0.5) 근처입니다]')
    print('  30장으로는 규칙을 못 찾았다는 뜻입니다. 위 표에 "한쪽으로만 답함" 표시가')
    print('  붙어 있으면 아무것도 구분하지 못한 상태입니다.')

_pa = RES[('A', 42)]['p']
_pc = RES[('C', 42)]['p']
_diff_ac = sum(1 for i in range(len(_pa)) if (_pa[i] >= 0.5) != (_pc[i] >= 0.5))
print()
print(f'[A 와 C 를 장별로 비교] {len(_pa)}장 중 다르게 답한 것 {_diff_ac}장 (seed 42)')
if _diff_ac <= 3:
    print('  거의 같습니다. 내 30장은 230장의 13% 인데 결과를 거의 바꾸지 못했습니다.')
    print('  많은 쪽이 이깁니다. 소수의 다른 판단이 묻히는 방식이 이것입니다.')
else:
    print('  내 30장이 결과를 움직였습니다. 어느 쪽으로 움직였는지 위 표에서 확인하세요.')

### 표를 읽는 법

**어떤 모델이 1등인지 보는 표가 아닙니다.** 어디서 갈리는지를 보는 표입니다.

```
  명확 T-EASY 칸    세 모델이 비슷할 겁니다      -> 쉬운 것은 누구나 맞힙니다
  애매 T-HARD 칸    여기가 벌어집니다            -> 차이는 어려운 것에서만 드러납니다
  A 와 C            내 30장을 더 넣으면 달라지나  -> 내 30장은 230장의 13% 입니다
```

### B 가 낮게 나왔다면

**"내 라벨이 나쁘다" 는 뜻이 아닙니다.** B 가 낮을 수 있는 이유는 최소 셋입니다.

```
  ① 라벨을 잘못 붙였다
  ② 30장은 200장보다 원래 너무 적다
  ③ 내 30장은 '모델이 가장 헷갈린 것' 만 골라 모은 것이라,
     전형적인 Donut 도 전형적인 Scratch 도 한 장이 없다
```

③ 을 특히 보십시오. 그 30장은 무작위로 뽑은 것이 아니라 **경계선만 골라 모은 것**입니다.
경계선만 보고 배운 모델은 정작 쉬운 것을 틀립니다. **B 가 낮은 것은 상당 부분 예정된
결과**이고, 이 표만으로는 ①②③ 을 가릴 수 없습니다.

### 0.5 아래로 내려갔다면

동전을 던져도 0.5 는 나옵니다. **0.5 아래는 못 맞힌 것이 아니라 뒤집힌 것입니다.**
내가 Donut 이라고 부른 모양을 채점셋은 Scratch 라고 부르고 있다는 뜻입니다.
**모양은 구분했고, 이름이 반대로 붙은 것입니다.** 이것은 실력 문제가 아니라
**용어를 서로 다르게 쓰고 있다**는 신호이고, 실제 현장에서 자주 일어나는 종류의 사고입니다.

### 괄호 안의 폭도 보세요

지난주에는 seed 를 바꾸면 정확도가 크게 흔들렸습니다. 이번에는 폭이 좁게 나올 겁니다.
**채점셋을 고정했기 때문입니다.** 지난주의 그 흔들림은 상당 부분
"매번 다른 이미지로 채점해서" 생긴 것이었습니다.

In [ ]:
# 어느 쪽으로 틀렸나 - 애매한 것(T-HARD)만 놓고 세 모델을 나란히 (seed 42)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, (_k, _label, _) in zip(axes, MODELS):
    _s = score(RES[(_k, 42)], T_HARD)
    ax.imshow(_s['cm'], cmap='Blues')
    ax.set_xticks(range(2)); ax.set_xticklabels(CLASS_NAMES, fontsize=8)
    ax.set_yticks(range(2)); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, _s['cm'][i, j], ha='center', va='center')
    ax.set_title(f"{_k}  acc {_s['acc']:.2f}", fontsize=10)
    ax.set_xlabel('Predicted', fontsize=8)
axes[0].set_ylabel('True (folder)', fontsize=8)
fig.suptitle('T-HARD only  (seed 42)', fontsize=11)
plt.tight_layout(); plt.show()

print('한 줄이 통째로 비어 있는 모델이 있습니까.')
print('그 모델은 모든 이미지에 같은 답을 냈습니다. 정확도가 나와도 아무것도 구분하지 못한 것입니다.')

---
## 무엇을 틀렸나 — 세 종류로 나눠 봅니다

**"몇 개 틀렸다" 가 아니라 "어떤 종류로 틀렸다" 를 말할 수 있어야 합니다.**

```
  ① 명확에서 틀림    누가 봐도 같은 답인데 틀림     -> 우리 쪽 문제. 질문·전처리를 먼저 본다
  ② 애매에서 갈림    사람도 답이 갈리는 이미지      -> 자연스러운 일. 왜 갈리는지 설명한다
  ③ 오라벨 의심      모델이 확신하며 라벨과 반대     -> 모델이 아니라 정답이 틀렸을 수 있다
```

③ 은 **라벨이 붙어 있는 데이터에서만** 할 수 있는 판정입니다.
여러분이 나눈 30장에는 정답이 없으므로 거기서는 오라벨을 말할 수 없습니다.
채점셋 50장에는 라벨이 붙어 있어서 볼 수 있습니다.

> 그 라벨도 **누군가 붙인 것**입니다. 공개 데이터셋을 만든 사람이 붙였습니다.
> ③ 에 걸린 이미지를 직접 보고, 붙어 있는 라벨과 모델 중 누가 더 그럴듯한지 판단해 보세요.

> 이 세 가지는 뒤에서 **AI 결과를 볼 때도 그대로** 씁니다.
> AI 가 ① 에서 틀렸다면 문제이고, ② 에서 갈렸다면 자연스러운 일입니다.

In [ ]:
# 세 종류로 나눠 봅니다 (기준 모델 A, seed 42)
_sa = score(RES[('A', 42)])
_sb = score(RES[('B', 42)])
_files = _sa['files']
assert _files == _sb['files']

_conf = {f: _pbar[i] for i, f in enumerate(TEST_FILES)}     # 모델 5개의 평균 확률
_yof = {f: TEST_Y[i] for i, f in enumerate(TEST_FILES)}

_clear_wrong = [i for i, f in enumerate(_files)
                if f in T_EASY and _sa['pred'][i] != _sa['y'][i]]
_amb_wrong = [i for i, f in enumerate(_files)
              if f in T_HARD and _sa['pred'][i] != _sa['y'][i]]
# 오라벨 의심 - 모델 5개 평균이 확신(0.8 이상)하는데 붙어 있는 라벨과 반대
MISLABEL = [f for f in _files
            if abs(_conf[f] - 0.5) > 0.3 and (0 if _conf[f] >= 0.5 else 1) != _yof[f]]

print(f'채점 {len(_files)}장 중 A 모델이 틀린 것')
print(f'  ① 명확에서 틀림   {len(_clear_wrong):3d}장')
print(f'  ② 애매에서 갈림   {len(_amb_wrong):3d}장')
print(f'  ③ 오라벨 의심     {len(MISLABEL):3d}장   (모델 5개가 확신하며 라벨과 반대)')

def show_bucket(idxs, title, n=6):
    idxs = list(idxs)[:n]
    print()
    if not idxs:
        print(title, '- 없습니다.')
        return
    print(title)
    fig, axes = plt.subplots(1, len(idxs), figsize=(2.5 * len(idxs), 3.4))
    if len(idxs) == 1:
        axes = [axes]
    for ax, i in zip(axes, idxs):
        f = _files[i]
        ax.imshow(Image.open(test_path(f)).convert('RGB'), interpolation='nearest')
        # label = 붙어 있는 라벨, A = 배포로 배운 모델, p = 모델 5개의 평균 확률
        ax.set_title(f'label {CLASS_NAMES[_sa["y"][i]]}   A {CLASS_NAMES[_sa["pred"][i]]}'
                     f'\np(Donut)={_conf[f]:.2f}', fontsize=8)
        ax.axis('off')
    plt.tight_layout(); plt.show()

if not _clear_wrong and not MISLABEL:
    print()
    print('① 과 ③ 이 0장입니다. 이 채점셋에서는 A 가 명확한 것을 다 맞혔고,')
    print('   붙어 있는 라벨과 확신을 갖고 부딪힌 것도 없다는 뜻입니다 - 라벨이 깨끗합니다.')
    print('   ③ 이 여러 장 나오는 데이터를 만나면, 모델보다 정답표를 먼저 의심해야 합니다.')

show_bucket(_clear_wrong, '[①] 명확한데 틀린 것 - 여기가 나오면 우리 쪽을 먼저 의심합니다')
show_bucket(_amb_wrong, '[②] 애매해서 갈린 것 - 사람도 갈립니다')
show_bucket([i for i, f in enumerate(_files) if f in MISLABEL],
            '[③] 오라벨 의심 - 붙어 있는 라벨 쪽이 이상해 보이는 것')

# ---- 같은 채점셋이므로 모델끼리도 장별로 비교됩니다 ----
_wa = {i for i in range(len(_files)) if _sa['pred'][i] != _sa['y'][i]}
_wb = {i for i in range(len(_files)) if _sb['pred'][i] != _sb['y'][i]}
print()
print(f'A(배포 200장) 와 B(내 라벨 30장) 를 장별로 - 애매 구간 {len(T_HARD)}장에서')
_h = {i for i, f in enumerate(_files) if f in T_HARD}
print(f'  둘 다 틀림   {len(_wa & _wb & _h):3d}장    이미지 자체가 어려운 것')
print(f'  B 만 틀림    {len((_wb - _wa) & _h):3d}장    <- 내 라벨 때문에 갈린 것')
print(f'  A 만 틀림    {len((_wa - _wb) & _h):3d}장    내 라벨이 오히려 맞춘 것')

---
# PART 3 — AI 는 이 이미지를 뭐라고 설명할까

**순서를 지키세요.**

```
  ① 이미지를 크게 본다  →  ② 내 판정을 적는다  →  ③ AI 준비  →  ④ AI 에게 묻는다
```

AI 답을 먼저 보면 거기에 끌려갑니다. **내 판단을 먼저 글로 박아두는 것**이 핵심입니다.

> **여기서는 한 번만 던져 봅니다.** 질문 유형 네 가지(열린 질문 · 구조 지정 · 선택지 제공 ·
> 근거 강제)의 **개념은 강의 슬라이드**에서 보고, **같은 이미지 3장에 네 유형을 넣어 표로
> 정리하는 것은 오늘 함께 받은 Gemini 노트북 PART 5 과제**입니다.
> 이 PART 3 에서 재는 것은 답의 *내용*이 아니라 **답이 매번 같은가(재현성)** 입니다.

> 키는 Colab 왼쪽 **열쇠 아이콘 🔑** 에 `GEMINI_API_KEY` 로 등록해 두세요.
> **노트북 액세스** 토글을 켜야 합니다.

In [ ]:
# ① 내가 나누면서 가장 망설였던 이미지를 봅니다 (AI 키가 없어도 돌아갑니다)
def show_big(path, scale=8):
    """작은 웨이퍼맵을 키워서 봅니다. NEAREST 를 쓰는 이유는 앞에서와 같습니다."""
    im = Image.open(path).convert('RGB')
    print(os.path.basename(path), '|', im.size, '(가로 x 세로 die 수)')
    plt.figure(figsize=(5, 5))
    plt.imshow(im, interpolation='nearest'); plt.axis('off'); plt.show()
    return im

# 어느 장이 가장 애매한지는 모델에게 물어보면 됩니다.
# 배포 200장으로 배운 모델이 0.5 에 가장 가까운 확률을 준 장이 가장 헷갈린 것입니다.
_pick = fit_predict(POOL_TRAIN, MY30, seed=42)
_rank = sorted(zip(_pick['files'], _pick['p']), key=lambda t: abs(t[1] - 0.5))
print('모델이 가장 헷갈린 순서 (내 30장 중)')
for _f, _p in _rank[:5]:
    print(f'  {_f:16} p(Donut)={_p:.3f}')
print(f'  ...')
print(f'  {_rank[-1][0]:16} p(Donut)={_rank[-1][1]:.3f}   <- 가장 확실해한 장')

MY_FILE = _rank[0][0]     # ← 가장 애매한 장. 1회차에 망설인 파일이 있으면 그 이름으로 바꾸세요.

TARGET = os.path.join(MIXED_DIR, MY_FILE)
if not os.path.isfile(TARGET):
    _have = sorted(os.path.basename(f) for f in glob.glob(os.path.join(MIXED_DIR, '*.*')))
    print('[확인 필요]', MY_FILE, '이 없습니다. 아래에서 골라 MY_FILE 에 넣으세요:')
    print('  ', ', '.join(_have[:12]), '...')
    raise FileNotFoundError(MY_FILE)

print()
print(f'오늘 볼 이미지: {MY_FILE}')
_ = show_big(TARGET)

In [ ]:
# ② 위 이미지를 보고 내 판정을 적습니다. 적은 뒤 이 셀을 실행하세요.
MY_CALL   = 'Donut'                 # ← 'Donut' 또는 'Scratch'
MY_REASON = '가운데가 비어 보여서'      # ← 왜 그렇게 봤는지 한 줄

assert MY_CALL in CLASS_NAMES, f'{CLASS_NAMES} 중 하나로 적으세요'
print('적었습니다 →', MY_CALL, '|', MY_REASON)

In [ ]:
# AI 준비 - 한 번만 실행하면 됩니다
PROVIDER = 'gemini'        # ← 'gemini' 또는 'openai' 로 바꿀 수 있습니다
API_KEY  = ''              # ← 보통 비워둡니다. 보안 비밀을 못 쓸 때만 여기에 붙여넣으세요.

import base64, io as _io

def _get_key(name):
    """Colab 왼쪽 열쇠 아이콘(보안 비밀)에서 키를 읽습니다.

    못 읽으면 입력창을 띄우지 않고 바로 멈춥니다.
    입력창을 띄우면 그것을 못 본 사람은 셀이 영원히 도는 것처럼 보입니다."""
    if API_KEY.strip():
        return API_KEY.strip()
    try:
        from google.colab import userdata
        k = userdata.get(name)
        if k:
            return k
        why = '값이 비어 있습니다'
    except Exception as e:
        why = type(e).__name__
    raise RuntimeError(
        f'{name} 를 읽지 못했습니다 ({why}).\n'
        '  1. 왼쪽 열쇠 아이콘 🔑 클릭\n'
        f'  2. 이름을 정확히  {name}  으로 새 보안 비밀 추가\n'
        '  3. [노트북 액세스] 토글 켜기   ← 이걸 빼먹는 경우가 가장 많습니다\n'
        '  4. 이 셀을 다시 실행\n'
        '  (그래도 안 되면 이 셀 맨 위 API_KEY 에 키를 직접 붙여넣으세요)')

def _big(im, scale=8):
    """웨이퍼맵은 아주 작습니다. 키워서 보내야 AI 가 볼 것이 생깁니다.
    NEAREST 를 쓰는 이유는 앞에서와 같습니다 - 없던 중간색이 생기면 안 됩니다."""
    im = im.convert('RGB')
    return im.resize((im.width * scale, im.height * scale), Image.NEAREST)

AI_MODEL = None

if PROVIDER == 'gemini':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'google-genai'],
                   capture_output=True, text=True)
    from google import genai
    from google.genai import types as _gt
    os.environ['GEMINI_API_KEY'] = _get_key('GEMINI_API_KEY')
    _client = genai.Client()
    _avail = [m.name.split('/')[-1] for m in _client.models.list()]
    for _n in ['gemini-3.6-flash', 'gemini-3.5-flash', 'gemini-2.5-flash']:
        if _n in _avail:
            AI_MODEL = _n; break
    if AI_MODEL is None:
        AI_MODEL = next((m for m in _avail if 'flash' in m), None)

    def ask_ai(path, prompt):
        r = _client.models.generate_content(
            model=AI_MODEL,
            contents=[prompt, _big(Image.open(path))],
            config=_gt.GenerateContentConfig(
                http_options=_gt.HttpOptions(timeout=60_000)))
        return r.text

else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'openai'],
                   capture_output=True, text=True)
    from openai import OpenAI
    _client = OpenAI(api_key=_get_key('OPENAI_API_KEY'))

    def ask_ai(path, prompt):
        buf = _io.BytesIO(); _big(Image.open(path)).save(buf, format='PNG')
        url = 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()
        last = None
        for name in ['gpt-5', 'gpt-4.1', 'gpt-4o']:
            try:
                r = _client.chat.completions.create(model=name, messages=[{
                    'role': 'user',
                    'content': [{'type': 'text', 'text': prompt},
                                {'type': 'image_url', 'image_url': {'url': url}}]}])
                globals()['AI_MODEL'] = name
                return r.choices[0].message.content
            except Exception as e:
                last = e
        raise RuntimeError(f'쓸 수 있는 모델을 찾지 못했습니다: {type(last).__name__}')

print('AI 준비 완료 -', PROVIDER, AI_MODEL or '(첫 호출 때 정해집니다)')

In [ ]:
# ④ AI 에게 던질 질문입니다. ★ 따옴표 안쪽을 바꿔가며 실험해 보세요.
PROMPT = ('이 이미지는 반도체 웨이퍼 맵입니다. '
          '시안색이 정상 die, 마젠타색이 불량 die, 흰색은 웨이퍼 바깥입니다.\n'
          '먼저 이미지에서 관찰되는 사실만 3가지를 쓰고, '
          '그 다음 줄에 Donut 인지 Scratch 인지 한 단어로 쓰세요.\n'
          '판단이 어려우면 "애매함"이라고 답해도 됩니다.')

import time as _time

print('[내가 던진 질문]')
print(PROMPT)
print()
print('AI 에게 보내는 중...', end=' ')
_t0 = _time.time()
_answer = ask_ai(TARGET, PROMPT)
print(f'{_time.time() - _t0:.0f}초 걸렸습니다')

# 답의 뒤쪽부터 훑어 결론 한 단어를 찾습니다
_lines = [l.strip() for l in _answer.splitlines() if l.strip()]
_ai_call = None
for _l in reversed(_lines):
    for _c in list(CLASS_NAMES) + ['애매함']:
        if _c.lower() in _l.lower():
            _ai_call = _c
            break
    if _ai_call:
        break

# 결론 줄은 관찰에서 빼고 보여줍니다
_obs = _lines[:-1] if (_ai_call and _lines and _ai_call.lower() in _lines[-1].lower()) else _lines

print()
print('=' * 60)
print('[AI 가 말한 관찰]')
for _l in _obs:
    print('  ', _l)

print()
print('AI 판정 :', _ai_call or '(한 단어로 답하지 않았습니다)')
print('내 판정 :', MY_CALL, '  |  ', MY_REASON)
print()

if _ai_call is None:
    print('-> AI 가 결론을 한 단어로 주지 않았습니다. 프롬프트를 조여 보세요.')
elif _ai_call == '애매함':
    print('-> AI 는 판단을 미뤘습니다. 여러분은 하나를 골라야 했습니다.')
    print('   그 차이가 무엇을 뜻하는지 생각해 보세요.')
elif _ai_call == MY_CALL:
    print('-> 같습니다. 그렇다고 둘 다 맞는다는 뜻은 아닙니다.')
    print('   위 관찰 3가지가 실제로 이미지에 있는지 확인하세요.')
else:
    print('-> 갈렸습니다. 누구 근거가 더 이미지에 붙어 있습니까?')

### 답을 읽을 때 볼 것

- AI 가 말한 **관찰 3가지가 실제로 이미지에 있습니까?** 없는 것을 지어내지 않았나요.
- 결론이 내 판정과 **같은가요, 다른가요.** 다르다면 누구 근거가 더 이미지에 붙어 있나요.
- AI 는 **"모르겠다"고 말하지 않습니다.** 확신 없이도 하나를 고릅니다.
  문장이 유창한 것과 맞는 것은 아무 상관이 없습니다.

> `PICK` 이나 파일 이름을 바꿔가며 두세 장 더 돌려보세요.

---
## 같은 질문을 세 번 — 답이 매번 같습니까

**정확도를 자랑하지 마세요.** 정답이 없는 데이터에서 "AI 정확도 90%" 같은 숫자는
쓸 수 없고, 써도 면접에서 무너집니다.

대신 **과장 없이 쓸 수 있는 숫자**를 남깁니다. 그중 하나가 **재현성**입니다.

```
  같은 이미지 · 같은 질문 · 3회 반복  ->  결론이 몇 번 같았는가
```

열린 질문은 매번 다른 답이 나오고, 근거를 강제한 질문은 잘 흔들리지 않습니다.
**그 차이가 프롬프트를 설계하는 이유이고, 그것이 여러분이 제출할 숫자입니다.**

> 12번 호출합니다. 1~2분 걸립니다.

In [ ]:
# 같은 이미지에 네 유형을 3번씩 - 결론이 몇 번 같은가 (재현성)
import time as _time
from collections import Counter as _Counter

PROMPTS = {
    '열린 질문':   '이 이미지를 분류해줘.',

    '구조 지정':   '이 웨이퍼 맵의 불량 die 분포를 중심-외곽 축으로 기술해줘. '
                   '마지막 줄에 Donut 또는 Scratch 로 결론을 쓰세요.',

    '선택지 제공': '이 웨이퍼 맵은 Donut 입니까 Scratch 입니까. '
                   '다른 말 없이 한 단어로만 답하세요.',

    '근거 강제':   '이 이미지는 반도체 웨이퍼 맵입니다. '
                   '시안색이 정상 die, 마젠타색이 불량 die, 흰색은 웨이퍼 바깥입니다.\n'
                   '먼저 이미지에서 관찰되는 사실만 3가지를 쓰고, '
                   '그 다음 줄에 Donut 인지 Scratch 인지 한 단어로 쓰세요.\n'
                   '판단이 어려우면 "애매함"이라고 답해도 됩니다.',
}
import unicodedata as _ud

def _pad(s, w):
    """한글은 화면에서 두 칸을 차지합니다. 그 폭으로 맞춥니다."""
    n = sum(2 if _ud.east_asian_width(ch) in 'WF' else 1 for ch in s)
    return s + ' ' * max(0, w - n)

N_REPEAT = 3

def conclude(text):
    """답의 뒤쪽부터 훑어 결론 한 단어를 찾습니다."""
    for line in reversed([l.strip() for l in text.splitlines() if l.strip()]):
        for c in list(CLASS_NAMES) + ['애매함']:
            if c.lower() in line.lower():
                return c
    return '(결론 없음)'

def reasons(text):
    """결론 줄을 뺀 나머지 - AI 가 든 근거입니다."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    body = lines[:-1] if len(lines) > 1 else lines
    return ' / '.join(body)[:400]

REPEAT_RESULT, _times = {}, []
for _name, _p in PROMPTS.items():
    _answers, _reasons = [], []
    print(_pad(_name, 12), end=' ')
    for _ in range(N_REPEAT):
        _t0 = _time.time()
        try:
            _txt = ask_ai(TARGET, _p)
            _answers.append(conclude(_txt))
            _reasons.append(reasons(_txt))
        except Exception as e:
            _answers.append(f'(실패: {type(e).__name__})')
            _reasons.append('')
        _times.append(_time.time() - _t0)
        print('.', end='')
    _top = _Counter(_answers).most_common(1)[0][1]
    _uniq = len({r for r in _reasons if r})
    REPEAT_RESULT[_name] = {'answers': _answers, 'agree': _top,
                            'reasons': _reasons, 'uniq_reasons': _uniq}
    print(_pad(' / '.join(_answers), 30)
          + f'  결론 {_top}/{N_REPEAT}   근거 {_uniq}가지')

AI_SEC = sum(_times) / len(_times)
print(f'\n평균 응답 시간 {AI_SEC:.1f}초')
print()

_all_same = all(r['agree'] == N_REPEAT for r in REPEAT_RESULT.values())
if _all_same:
    print('네 유형 모두 결론이 3/3 으로 같게 나왔습니다. 흔한 일입니다.')
    print('그렇다고 네 질문이 같은 질문이라는 뜻은 아닙니다. 오른쪽 "근거 n가지" 를 보세요.')
    print('결론은 같은데 근거가 매번 다르다면, 그 결론은 근거에서 나온 것이 아닙니다.')
else:
    print('유형마다 흔들림이 다릅니다. 그것이 프롬프트를 설계하는 이유입니다.')

print()
print('■ 마지막 실행에서 AI 가 든 근거')
for _name, _r in REPEAT_RESULT.items():
    _last = _r['reasons'][-1] if _r['reasons'] else ''
    print(f'  [{_name}] {_last[:110] if _last else "(근거 없이 결론만)"}')
print()
print('여기서 진짜 볼 것은 "몇 번 같았나" 가 아니라')
print('  ① 결론이 같아도 근거가 매번 다른가')
print('  ② 그 근거가 이미지에 실제로 있는가   <- 이건 사람만 확인할 수 있습니다')
print('입니다. ② 를 확인해서 제출 표의 마지막 칸에 적으세요.')

In [ ]:
# 효율 지표 정리 - 과장 없이 적을 수 있는 숫자만 남깁니다
MY_LABELING_MIN = 0     # ← 지난주에 30장을 눈으로 나누는 데 걸린 시간(분). 기억나는 대로.

print('[1] 재현성 - 질문 유형별로 결론이 몇 번 같았나')
for _name, _r in REPEAT_RESULT.items():
    print('    ' + _pad(_name, 12) + f'{_r["agree"]}/{N_REPEAT}')

print()
print('[2] 라벨링 소요 시간')
if MY_LABELING_MIN > 0:
    _ai_min = AI_SEC * len(MIXED_FILES) / 60
    print(f'    눈으로 30장    {MY_LABELING_MIN}분')
    print(f'    AI 응답 30장   {_ai_min:.1f}분   (응답 시간만. 사람이 읽고 판단하는 시간은 별도)')
    print()
    print('    AI 쪽이 짧더라도 "N배 빨라졌다" 고 쓰지 마세요.')
    print('    AI 답을 그대로 쓸 수 없고 사람이 검토해야 하므로, 정직한 문장은 이것입니다:')
    print(f'      "30장 육안 분류 {MY_LABELING_MIN}분 / AI 응답 {_ai_min:.1f}분 + 검토 시간"')
else:
    print('    위 MY_LABELING_MIN 에 지난주 걸린 시간(분)을 적고 다시 실행하세요.')

print()
print('[3] 원인 후보 축소 폭 - 이것은 노트북이 아니라 여러분이 채웁니다')
print('    "파라미터 후보 3개 -> 1개로 좁힘, 판별에 필요한 데이터 1종 특정" 같은 형태로')
print('    제출 PPT 에 적으세요. 이 세 가지가 정확도 대신 쓸 수 있는 숫자입니다.')

---
## 제출 PPT 초안을 AI 로 만들기

배포된 **템플릿을 그대로 받아서** AI 가 채우고, 채워진 pptx 를 내려받습니다.
구글 슬라이드에 다시 올려서 고치시면 됩니다.

```
  ① 내 정보를 적는다      <- 여기만 손으로 채웁니다 (세 칸)
        ↓
  ② 실행만 한다           <- 채울 것 없습니다. 그냥 실행
        ↓
  ③ 내려받아 내가 고친다
        ↓
  ④ 고친 곳 3군데를 적는다  <- 평가 대상
```

**④ 가 이 과제의 핵심입니다.** 평가하는 것은 AI 를 썼는지가 아니라 **AI 를 어디서 이겼는지** 입니다.

AI 가 쓴 문장은 매끄럽습니다. 그런데 이미지에 붙어 있지 않으면 반박 한 번에 무너집니다.
고친 곳 세 군데가 그 차이를 보여주는 증거입니다.

> AI 는 여러분이 보지 않은 것을 지어냅니다. **초안을 그대로 내면 5분 안에 드러납니다.**

In [ ]:
# ① 내 정보를 적고 템플릿을 받습니다
TEMPLATE_URL = 'https://docs.google.com/presentation/d/1Gepwx_vLdSg3frFWvI5SqxpzF0DtB6vu/edit?usp=sharing'

# ---- 여기 세 칸만 채우세요 ----
MY_PROCESS   = ''      # 공정 - ETCH / CMP / THIN FILM
MY_PARAMETER = ''      # 내가 범인으로 지목한 파라미터 (예: 포커스링 소모)
MY_OBSERVED  = ''      # 웨이퍼맵에서 내가 '실제로 본 것' 을 한두 문장으로

# AI 에게 더 하고 싶은 말이 있으면 적으세요. 비워 두어도 됩니다.
# (여기 쓴 내용은 '추가 지시' 로 들어갑니다. 템플릿을 채우는 규칙은 그대로 유지됩니다.)
MY_PPT_PROMPT = """

"""
# ------------------------------------------------------------------
import subprocess, sys, urllib.request
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-pptx'],
               capture_output=True, text=True)
from pptx import Presentation

_m = re.search(r'/presentation/d/([A-Za-z0-9_-]{20,})', TEMPLATE_URL)
if not _m:
    raise ValueError('템플릿 링크에서 문서 ID 를 찾지 못했습니다.')
TEMPLATE_PPTX = '/content/template.pptx'
urllib.request.urlretrieve(
    f'https://docs.google.com/presentation/d/{_m.group(1)}/export/pptx', TEMPLATE_PPTX)

_tpl = Presentation(TEMPLATE_PPTX)
print(f'템플릿을 받았습니다 - {len(_tpl.slides)}장')

TEMPLATE_GUIDE = []
for _i, _s in enumerate(_tpl.slides, 1):
    _lines = [t for sh in _s.shapes if sh.has_text_frame
              for t in [sh.text_frame.text.strip()] if t]
    if _lines:
        TEMPLATE_GUIDE.append(f'[{_i}장] ' + ' / '.join(l.replace(chr(10), ' ')[:70] for l in _lines[:4]))
        print(f'  {_i}장  {_lines[1][:52] if len(_lines) > 1 else _lines[0][:52]}')

if not all([MY_PROCESS, MY_PARAMETER, MY_OBSERVED]):
    print()
    print('[확인 필요] 위 세 칸을 채우고 이 셀을 다시 실행하세요.')
else:
    print()
    print(f'{MY_PROCESS} · {MY_PARAMETER}')

In [ ]:
# 준비 - 읽지 않으셔도 됩니다. 실행만 하세요.
# (템플릿의 어느 칸에 무엇을 넣을지 적어둔 표와, AI 에게 묻는 함수들입니다)
import json as _json

def ask_text(prompt):
    """이미지 없이 글만 물어봅니다."""
    if PROVIDER == 'gemini':
        r = _client.models.generate_content(
            model=AI_MODEL, contents=[prompt],
            config=_gt.GenerateContentConfig(http_options=_gt.HttpOptions(timeout=180_000)))
        return r.text
    r = _client.chat.completions.create(
        model=AI_MODEL, messages=[{'role': 'user', 'content': prompt}])
    return r.choices[0].message.content

# 채울 칸 - (슬라이드 번호, 도형 이름, 키, 설명)
SLOTS = [
    (2, 'Text 12', 'claim',       '한 줄 주장. 파라미터·위치·모양이 한 문장에 다 들어갈 것'),
    (2, 'Text 15', 'weak_short',  '가장 약한 고리를 한 줄로'),
    (3, 'Text 6',  'prompt_note', '네 유형 질문에서 무엇을 알았나. 결론이 아니라 검증 가능성의 차이를 쓸 것'),
    (4, 'Text 6',  'process',     '내 공정이 웨이퍼에 무엇을 하는지 두 문장'),
    (4, 'Text 11', 'p1_title',    '균일도가 깨지는 지점 1 - 짧은 제목'),
    (4, 'Text 12', 'p1_body',     '지점 1 설명 두 문장'),
    (4, 'Text 16', 'p2_title',    '지점 2 - 짧은 제목'),
    (4, 'Text 17', 'p2_body',     '지점 2 설명 두 문장'),
    (4, 'Text 21', 'p3_title',    '지점 3 - 짧은 제목'),
    (4, 'Text 22', 'p3_body',     '지점 3 설명 두 문장'),
    (5, 'Text 6',  'param',       '지목 파라미터 이름'),
    (5, 'Text 10', 'where',       '이 파라미터가 범인이면 결함이 나타날 위치'),
    (5, 'Text 13', 'shape',       '나타날 모양'),
    (5, 'Text 16', 'trend',       '시간이 지나면 어떻게 변하는가'),
    (5, 'Text 19', 'cosymptom',   '함께 보여야 할 동반 증상'),
    (5, 'Text 22', 'why',         '왜 이 파라미터인가 - 두세 문장'),
    (5, 'Text 26', 'compare',     '내 예측과 실제 관찰이 어긋나는 지점. 어긋남을 숨기지 말 것'),
    (6, 'Text 6',  'me_param',    '내 파라미터 이름'),
    (6, 'Text 9',  'me_where',    '내 예측 위치 - 짧게'),
    (6, 'Text 11', 'me_shape',    '내 예측 모양 - 짧게'),
    (6, 'Text 13', 'me_trend',    '내 예측 추세 - 짧게'),
    (6, 'Text 16', 'a_param',     '같은 조 다른 파라미터 A 이름'),
    (6, 'Text 19', 'a_where',     'A 의 예측 위치 - 짧게'),
    (6, 'Text 21', 'a_shape',     'A 의 예측 모양 - 짧게'),
    (6, 'Text 23', 'a_trend',     'A 의 예측 추세 - 짧게'),
    (6, 'Text 26', 'b_param',     '같은 조 다른 파라미터 B 이름'),
    (6, 'Text 29', 'b_where',     'B 의 예측 위치 - 짧게'),
    (6, 'Text 31', 'b_shape',     'B 의 예측 모양 - 짧게'),
    (6, 'Text 33', 'b_trend',     'B 의 예측 추세 - 짧게'),
    (6, 'Text 36', 'discrim',     '세 가설을 가를 수 있는 결정적 측정 하나'),
    (7, 'Text 7',  'c1_title',    'Fab 에서 확인할 것 1 - 제목'),
    (7, 'Text 8',  'c1_body',     '1 을 왜 보는가 한 문장'),
    (7, 'Text 14', 'c2_title',    '확인할 것 2 - 제목'),
    (7, 'Text 15', 'c2_body',     '2 를 왜 보는가'),
    (7, 'Text 21', 'c3_title',    '확인할 것 3 - 제목'),
    (7, 'Text 22', 'c3_body',     '3 을 왜 보는가'),
    (7, 'Text 28', 'c4_title',    '확인할 것 4 - 제목'),
    (7, 'Text 29', 'c4_body',     '4 를 왜 보는가'),
    (8, 'Text 6',  'w1_title',    '약한 고리 1 - 한 줄'),
    (8, 'Text 7',  'w1_body',     '약한 고리 1 설명 두세 문장'),
    (8, 'Text 10', 'w2_title',    '약한 고리 2 - 한 줄'),
    (8, 'Text 11', 'w2_body',     '약한 고리 2 설명 두세 문장'),
    (8, 'Text 14', 'strength',    '그래서 내 주장은 어느 강도까지 유지되는가'),
]


def build_prompt(slots):
    """학생이 쓴 지시는 '추가 지시' 로 넣고, JSON 규약은 항상 붙입니다.

    학생 프롬프트로 통째로 갈아치우면 AI 가 그냥 글로 답해서 채울 수가 없습니다."""
    extra = MY_PPT_PROMPT.strip() or '(없음)'
    keys = _json.dumps({k: d for _, _, k, d in slots}, ensure_ascii=False, indent=1)
    return f"""당신은 반도체 공정 엔지니어의 발표 자료 초안을 씁니다.

[담당 공정]  {MY_PROCESS}
[지목 파라미터]  {MY_PARAMETER}
[웨이퍼맵에서 실제로 본 것]  {MY_OBSERVED}

[내가 추가로 요청한 것]
{extra}

[템플릿이 요구하는 것]
{chr(10).join(TEMPLATE_GUIDE)}

규칙
1. 위 '실제로 본 것' 에 없는 관찰을 지어내지 마십시오. 필요하면 "확인 필요" 라고 쓰십시오.
2. 단정하지 마십시오. "유력하다" 까지만 쓰고 "이다" 라고 쓰지 마십시오.
3. compare 와 w1/w2 에는 내 주장에 불리한 내용을 반드시 쓰십시오.
4. 정확도 같은 수치를 성과로 쓰지 마십시오. 정답이 없는 데이터입니다.
5. title 계열은 20자 이내, body 계열은 두세 문장으로 짧게.

출력 형식 - 이것만은 반드시 지키십시오.
아래 키를 가진 JSON 객체 하나만 출력하십시오. 인사말·설명·표를 붙이지 마십시오.
첫 글자는 {{ 이고 마지막 글자는 }} 여야 합니다.

{keys}
"""

def parse_json(text):
    """코드블록이든 앞뒤에 말이 붙었든 JSON 만 뽑아냅니다."""
    t = (text or '').strip()
    cands = []
    if '```' in t:
        for part in t.split('```'):
            part = part.strip()
            if part.lower().startswith('json'):
                part = part[4:].strip()
            if part.startswith('{'):
                cands.append(part)
    i, j = t.find('{'), t.rfind('}')
    if i >= 0 and j > i:
        cands.append(t[i:j + 1])
    for c in cands:
        try:
            return _json.loads(c)
        except Exception:
            pass
    return None


def _set(slide, name, text):
    for sh in slide.shapes:
        if sh.has_text_frame and sh.name == name and sh.text_frame.paragraphs[0].runs:
            sh.text_frame.paragraphs[0].runs[0].text = str(text)
            for p in sh.text_frame.paragraphs[1:]:
                for r in p.runs:
                    r.text = ''
            return True
    return False


print('준비 완료')

In [ ]:
# ② 실행만 하면 됩니다. 이 셀에는 채울 것이 없습니다.
#    (채우는 칸은 바로 위 셀에 있습니다)

_empty = [n for n, v in [("MY_PROCESS", MY_PROCESS),
                         ("MY_PARAMETER", MY_PARAMETER),
                         ("MY_OBSERVED", MY_OBSERVED)] if not str(v).strip()]
if _empty:
    print('[확인 필요] 위 셀에서 아직 비어 있는 칸이 있습니다:')
    for _n in _empty:
        print('   ', _n)
    print()
    print('  위 셀로 올라가 따옴표 사이에 적고, 그 셀을 먼저 실행한 뒤 여기로 오세요.')
    raise RuntimeError('입력이 비어 있습니다.')

# 한 번에 43칸을 물어보면 답이 중간에 잘립니다. 두 번에 나눠 받습니다.
_batches = [[s for s in SLOTS if s[0] <= 5], [s for s in SLOTS if s[0] >= 6]]
DRAFT = {}
for _bi, _batch in enumerate(_batches, 1):
    _prompt = build_prompt(_batch)
    if _bi == 1:
        PPT_PROMPT_USED = _prompt
    _part = None
    for _attempt in (1, 2):
        print(f'AI 가 템플릿을 채우는 중... ({_bi}/2, {_attempt}번째 시도)', end=' ')
        _raw = ask_text(_prompt if _attempt == 1 else
                        'JSON 객체 하나만 출력하십시오. 다른 말은 한 글자도 쓰지 마십시오.\n\n' + _prompt)
        _part = parse_json(_raw)
        print('받았습니다' if _part else '-> JSON 이 아닙니다')
        if _part:
            break
    if not _part:
        print()
        print('[실패] AI 가 JSON 으로 답하지 않았습니다. 받은 답의 앞부분:')
        print(_raw[:400])
        print()
        print('  MY_PPT_PROMPT 에 "JSON 말고 설명해줘" 같은 지시가 들어 있지 않은지 보세요.')
        print('  비워 두고 다시 실행하면 대부분 해결됩니다.')
        raise RuntimeError('초안을 JSON 으로 받지 못했습니다.')
    DRAFT.update(_part)

print(f'채울 내용 {len(DRAFT)}칸을 받았습니다')

# ---- 템플릿에 채워 넣습니다 ----
_out = Presentation(TEMPLATE_PPTX)
_filled = _missing = 0
for _sn, _shape, _key, _ in SLOTS:
    if _key in DRAFT and _set(_out.slides[_sn - 1], _shape, DRAFT[_key]):
        _filled += 1
    else:
        _missing += 1

# 표지의 공정·파라미터는 내가 적은 값을 그대로 씁니다
for _shape, _val in [('Text 7', MY_PROCESS), ('Text 9', MY_PARAMETER)]:
    _set(_out.slides[1], _shape, _val)

# 이름 줄은 쓰지 않습니다 - 라벨과 값 두 도형을 지웁니다
for _shape in ('Text 4', 'Text 5'):
    for _sh in list(_out.slides[1].shapes):
        if _sh.name == _shape:
            _sh._element.getparent().remove(_sh._element)

# 안내 상자에 남은 '이름 ·' 도 지웁니다 (여러 줄짜리라 해당 줄만 손댑니다)
for _sh in _out.slides[1].shapes:
    if _sh.has_text_frame and _sh.name == 'Text 18':
        for _para in _sh.text_frame.paragraphs:
            for _r in _para.runs:
                if '이름' in _r.text:
                    _r.text = _r.text.replace('이름 · ', '').replace('이름·', '')

# 과제 ① 표는 PART 3 에서 실제로 잰 결과로 채웁니다
if 'REPEAT_RESULT' in dir():
    for _sh in _out.slides[2].shapes:
        if _sh.has_table:
            for _row in list(_sh.table.rows)[1:]:
                _name = _row.cells[0].text.strip()
                if _name in REPEAT_RESULT:
                    _ans = REPEAT_RESULT[_name]['answers']
                    _cell = _row.cells[2]
                    if _cell.text_frame.paragraphs[0].runs:
                        _cell.text_frame.paragraphs[0].runs[0].text = (
                            f'{_ans[0]}  (일치 {REPEAT_RESULT[_name]["agree"]}/{N_REPEAT})')
            print('과제 ① 표에 PART 3 의 재현성 결과를 넣었습니다')

# 안내 슬라이드(1장) 를 지우고, '예 시' 배지를 비웁니다
_ids = _out.slides._sldIdLst
_out.part.drop_rel(list(_ids)[0].rId)
_ids.remove(list(_ids)[0])
for _s in _out.slides:
    _set(_s, 'Text 1', '')

SUBMIT_PPTX = '/content/my_submission.pptx'
_out.save(SUBMIT_PPTX)
print(f'\n초안 완성 - 채운 칸 {_filled}개 / 못 찾은 칸 {_missing}개')
print(f'파일: {SUBMIT_PPTX}')
print()
print('이제 여러분 차례입니다.')
print('  1. 내려받아 열고, 지어낸 곳·틀린 곳을 고치십시오.')
print('  2. 고친 곳 3군데를 아래 셀에 적으십시오. 그것이 평가 대상입니다.')

if files is not None:
    files.download(SUBMIT_PPTX)

In [ ]:
# ③ AI 초안에서 내가 고친 곳 3군데 - 여기가 평가 대상입니다
#   ('AI 가 쓴 것', '내가 고친 것 - 왜 고쳤는지')
MY_FIXES = [
    ('', ''),
    ('', ''),
    ('', ''),
]

# ------------------------------------------------------------------
_p = (PPT_PROMPT_USED if 'PPT_PROMPT_USED' in dir() else '').strip()
_fixes = [(a.strip(), b.strip()) for a, b in MY_FIXES if a.strip() or b.strip()]

print('=' * 62)
print('[제출용] AI 로 만든 슬라이드에 붙일 기록')
print('=' * 62)

if not _p:
    print()
    print('[비어 있음] 위 셀을 먼저 실행하세요. 쓴 프롬프트가 자동으로 여기 담깁니다.')
else:
    print()
    print('■ 내가 쓴 프롬프트')
    for _l in _p.splitlines()[:14]:
        print('   ', _l)
    if len(_p.splitlines()) > 14:
        print('    ... (전문은 제출 슬라이드에 붙이세요)')

    _chk = [('관찰을 먼저, 결론은 나중에', any(k in _p for k in ('관찰', '먼저', '근거'))),
            ('선택지를 좁혀준다', any(k in _p for k in ('중 하나', '한 단어', '이내', '형식', 'JSON'))),
            ('"애매하다" 고 답할 여지를 준다', any(k in _p for k in ('애매', '모르', '어려우', '확인 필요')))]
    print()
    print('■ 세 가지 원칙 점검   (없다고 틀린 것은 아닙니다. 왜 뺐는지 말할 수 있으면 됩니다)')
    for _name, _ok in _chk:
        print(f'    [{"O" if _ok else " "}] {_name}')

print()
print(f'■ AI 초안에서 고친 곳   {len(_fixes)}/3')
if not _fixes:
    print('    [비어 있음] MY_FIXES 를 채우세요. 여기가 이 과제의 핵심입니다.')
for _i, (_a, _b) in enumerate(_fixes, 1):
    print(f'    {_i}. AI : {_a}')
    print(f'       나 : {_b}')
    if len(_b) < 10:
        print('       ^ 이유가 너무 짧습니다. "왜" 를 한 문장으로 쓰세요.')

print()
print('=' * 62)
if _p and len(_fixes) == 3 and all(len(b) >= 10 for _, b in _fixes):
    print('제출 준비 완료. 위 내용을 슬라이드 한 장으로 옮기세요.')
else:
    print('아직 빈 칸이 있습니다. 채우고 이 셀을 다시 실행하세요.')

## 오늘 실습에서 적어둘 것

> 3회차 제출물(A4 한 장 · 원인 지도)은 따로 받은 **양식과 치트시트**를 쓰십시오.
> 아래는 그 제출 PPT 에 들어갈 **오늘 실습의 기록**입니다.

**① 세 모델의 점수.** 특히 `애매 T-HARD` 칸 세 개.

**② 옆 사람과 B 를 비교해 보세요.**
데이터도 코드도 채점셋도 학습 횟수도 같습니다. 다른 것은 **폴더에 넣은 판단**뿐입니다.
왜 다른 숫자가 나왔을까요?

**③ B 가 A 보다 낮았다면 그 이유를 셋 중 무엇이라고 보십니까.**
(라벨 / 장수 / 이미지 난이도) **그리고 그것을 어떻게 확인하겠습니까.**
확인할 방법을 한 문장으로 쓰면 그게 실험 설계입니다.

**④ 강사 라벨과 갈린 장의 파일 이름**, 그리고 왜 그렇게 봤는지 한 줄씩.

**⑤ 틀린 것을 세 종류로 나눈 결과.** 명확에서 틀림 / 애매에서 갈림 / 오라벨 의심 각 몇 장.

**⑥ 효율 지표 세 개** — 제출 PPT 에 그대로 들어갑니다.

```
  재현성        질문 유형별 결론 일치 횟수 (예: 열린 질문 1/3 -> 근거 강제 3/3)
  소요 시간     30장 육안 분류 N분 / AI 응답 M분 + 검토 시간
  후보 축소     파라미터 후보 3개 -> 1개, 판별 데이터 1종 특정
```

**⑦ AI 로 슬라이드를 만들었다면** — 쓴 프롬프트 전문과 고친 곳 3군데.
위 셀이 정리해 준 것을 그대로 한 장에 옮기면 됩니다.

> **정확도 수치를 성과로 쓰지 마십시오.** 정답이 없는 데이터입니다.
> 위 세 가지는 과장이 아니면서 "효율을 높였다" 를 뒷받침합니다.

---

> **이 실습의 결론**
>
> 같은 코드, 같은 채점셋, 같은 학습 횟수인데 결과는 다르게 나옵니다.
> 데이터를 어떻게 나누느냐, 누가 라벨을 붙였느냐에 따라 숫자가 바뀝니다.
>
> 그래서 중요한 것은 높은 점수를 내는 일이 아니라,
> **그 숫자가 무엇을 뜻하는지 설명할 수 있는 일**입니다.

---
## 사용한 데이터

본 실습의 웨이퍼맵 이미지는 **공개 데이터셋**에서 가져온 것입니다.
포트폴리오나 자소서에 쓸 때는 공개 데이터셋임을 반드시 밝히십시오.
“실제 공정 데이터를 분석했다”고 쓰면 면접에서 문제가 됩니다.